# Phase 1 evidence audit

Tomorrow: edit and run **Configure paths**, then run **Run audit** once. Opening this notebook or editing paths does no work. The audit runs in one CPU-only child process; the kernel holds only its launch settings and streamed text.

Inputs are read in place. The worker writes `phase1_report.md` and `phase1_run.log` beside itself. Exit 2 means a partial report with missing/unverified evidence. Interrupting the run cell stops and waits for the child. After it exits, you can shut down this notebook's kernel. See `README.md` for input/provenance details and the PowerShell alternative.

In [ ]:
# Configure paths — this cell only defines paths; it does not launch the audit.
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\moht2\Documents\Afterimage")
EXTRA_INPUT_ROOTS = [
    # Path(r"D:\DR evidence\APTOS exports"),
    # Path(r"D:\DR evidence\Messidor-2"),
    # Path(r"D:\DR evidence\original images"),
]

# Change this only if you move the three audit files. Output stays here,
# independently of which reference project or input roots you inspect.
WORKER = Path(r"C:\Users\moht2\Documents\Afterimage\audit\phase1_audit.py")


In [ ]:
# Run audit — launch exactly one Python child, stream it, and reap it.
import os
import subprocess
import sys

worker = WORKER.expanduser().resolve()
if not worker.is_file():
    raise FileNotFoundError(f"Worker not found: {worker}")
report_path = worker.parent / "phase1_report.md"
command = [sys.executable, "-u", str(worker),
           "--project-root", str(PROJECT_ROOT.expanduser().resolve())]
for input_root in EXTRA_INPUT_ROOTS:
    command.extend(["--input-root", str(input_root.expanduser().resolve())])
environment = {**os.environ, "PYTHONIOENCODING": "utf-8",
               "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
               "CUDA_VISIBLE_DEVICES": "", "HIP_VISIBLE_DEVICES": ""}
child = None
interrupted = False
try:
    child = subprocess.Popen(
        command, cwd=worker.parent, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1, env=environment, shell=False,
    )
    for line in child.stdout:
        print(line, end="", flush=True)
    child.wait()
except BaseException:
    interrupted = True
    if child is not None:
        if child.poll() is None:
            try:
                child.terminate()
            except OSError:
                pass  # The process may have exited between poll and terminate.
            try:
                child.wait(timeout=15)
            except subprocess.TimeoutExpired:
                child.kill()
                child.wait()
        else:
            child.wait()
    raise
finally:
    if child is not None and child.stdout is not None:
        child.stdout.close()
    print(f"\nChild exit status: {child.returncode if child is not None else 'not launched'}", flush=True)
    print(f"Report path: {report_path}", flush=True)
    if interrupted:
        print("Run interrupted/failed; the report may be partial or from a previous run. Check its timestamp.", flush=True)
    elif child is not None:
        print({0: "Audit finished; review the report and provenance handoff.",
               2: "Partial audit; see missing/unverified inputs in the report."}
              .get(child.returncode, "Worker failed; inspect phase1_run.log and any partial report."), flush=True)
